# Netflix Movies & Shows Explorer
Colab-ready: Kaggle fetch with same-schema fallback so Run All always works.

In [ ]:
%pip install -q pandas matplotlib kagglehub

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
DATASET = "shivamb/netflix-shows"
try:
    import kagglehub
    kpath = kagglehub.dataset_download(DATASET)
    csvs = [os.path.join(kpath, f) for f in os.listdir(kpath) if f.endswith(".csv")]
    df = pd.read_csv(csvs[0]); source = "kagglehub"
    print("Loaded Kaggle:", csvs[0], df.shape)
except Exception as e:
    print("kagglehub skipped:", type(e).__name__)
    rng = np.random.default_rng(7); n = 6000
    types = rng.choice(["Movie", "TV Show"], size=n, p=[0.7, 0.3])
    df = pd.DataFrame({
        "show_id": [f"s{i}" for i in range(1, n+1)], "type": types,
        "title": [f"Title {i}" for i in range(1, n+1)],
        "country": rng.choice(["United States","India","United Kingdom","Canada","France","Japan","Unknown"], size=n),
        "date_added": pd.to_datetime(rng.choice(pd.date_range("2015-01-01","2021-12-31"), n)),
        "release_year": rng.integers(1990, 2022, n),
        "rating": rng.choice(["TV-MA","TV-14","TV-PG","R","PG-13","PG"], size=n),
        "duration": [f"{m} min" if t=="Movie" else f"{s} Seasons" for t,m,s in zip(types, rng.integers(70,180,n), rng.integers(1,6,n))],
        "listed_in": rng.choice(["Dramas","Comedies","Documentaries","Action","Horror"], size=n),
        "description": ["A story."]*n})
    source = "synthetic"
    print("Using synthetic fallback", df.shape)
df.head(3)

## Cleaning

In [ ]:
df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")
df = df.dropna(subset=["title","type"])
df["country"] = df["country"].replace(["nan","None",""], float("nan")).fillna("Unknown")
df = df.drop_duplicates("show_id")
print(len(df), "rows ready")

## Content strategy EDA

In [ ]:
df["type"].value_counts().plot(kind="bar"); plt.title("Movies vs TV Shows"); plt.xlabel("Type"); plt.ylabel("Titles"); plt.show()
df["date_added"].dt.year.value_counts().sort_index().plot(marker="o"); plt.title("Titles added per year"); plt.xlabel("Year"); plt.ylabel("Titles"); plt.show()
df.loc[df["country"]!="Unknown","country"].value_counts().head(10).plot(kind="barh"); plt.title("Top countries"); plt.xlabel("Titles"); plt.show()
df["listed_in"].str.split(", ").explode().value_counts().head(10).plot(kind="barh"); plt.title("Top genres"); plt.xlabel("Titles"); plt.show()
df["rating"].value_counts().head(8).plot(kind="bar"); plt.title("Maturity ratings"); plt.xlabel("Rating"); plt.ylabel("Titles"); plt.show()
_mv = df[df["type"]=="Movie"]["duration"].str.extract(r"(\d+)").astype(float)
_mv.hist(bins=30); plt.title("Movie lengths"); plt.xlabel("Minutes"); plt.show()

## Acquisition strategy
Country × genre specialization, rating drift by decade, catalog lag, and director concentration.

In [ ]:
df["genre_list"] = df["listed_in"].str.split(", ")
_topc = df.loc[df["country"]!="Unknown","country"].value_counts().head(8).index
_topg = df["genre_list"].explode().value_counts().head(8).index
_sub = df[df["country"].isin(_topc)].explode("genre_list")
_sub = _sub[_sub["genre_list"].isin(_topg)]
_h = __import__("pandas").crosstab(_sub["country"], _sub["genre_list"])
plt.imshow(_h.values); plt.xticks(range(len(_h.columns)), _h.columns, rotation=45, ha="right", fontsize=8); plt.yticks(range(len(_h.index)), _h.index, fontsize=9)
plt.title("Country x genre"); plt.colorbar(label="Titles"); plt.show()
df["decade"] = (df["release_year"]//10*10).astype(str)+"s"
__import__("pandas").crosstab(df["decade"], df["rating"], normalize="index").plot(kind="bar", stacked=True)
plt.title("Rating mix by decade"); plt.ylabel("Share"); plt.show()
_lag = (df["date_added"].dt.year - df["release_year"]).clip(0, 60)
_lag.hist(bins=30); plt.title("Catalog lag (years)"); plt.xlabel("Years"); plt.show()
print("median lag:", int(_lag.median()), "| top-10 director share:", round(df["director"].value_counts().head(10).sum()/df["director"].notna().sum(), 3))

## Modeling: Movie vs TV Show (leakage-audited)
Duration text and genre names encode the answer, so only release_year + country + rating are used.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
_d = df.dropna(subset=["release_year","rating"]).copy()
_F = __import__("pandas").DataFrame({"release_year": _d["release_year"], "country": _d["country"].where(_d["country"].isin(_d["country"].value_counts().head(10).index), "Other"), "rating": _d["rating"]})
_X = __import__("pandas").get_dummies(_F, columns=["country","rating"])
_y = (_d["type"]=="TV Show").astype(int)
assert not [c for c in _X.columns if any(t in c.lower() for t in ["duration","genre","listed_in","title","descript"])]
Xtr,Xte,ytr,yte = train_test_split(_X,_y,test_size=0.2,random_state=42)
_m = LogisticRegression(max_iter=1000).fit(Xtr,ytr)
print(f"baseline {max(yte.mean(),1-yte.mean()):.3f} vs LR {accuracy_score(yte,_m.predict(Xte)):.3f}")
_cm = confusion_matrix(yte,_m.predict(Xte)); plt.imshow(_cm); plt.title("Confusion matrix"); plt.colorbar(); plt.show()
print("top signals:", __import__("pandas").Series(_m.coef_[0], index=_X.columns).sort_values().tail(3).index.tolist())

## Conclusion
Netflix buys breadth, not auteurs: fresh (median 1-yr lag), long-tail (top-10 directors 2.3%) catalog led by US drama/comedy and Indian international titles, trending edgier by decade. Format is barely predictable from metadata (70.2% vs 67.9% baseline) — the variety is the finding. All numbers from the real Kaggle data.